# 🎓 Interactive Lesson: Building a Custom Search Engine from Scratch

---

## What You Will Learn

By the end of this notebook, you will understand how to:

| # | Concept | Real-World Analogy |
|---|---------|-------------------|
| 1 | Load documents from Google Drive | Opening a book from a shelf |
| 2 | Convert words into numbers (Word2Vec) | Giving every word a GPS coordinate |
| 3 | Search using Cosine Similarity | Finding the closest GPS location to yours |
| 4 | Summarize text with AI (BART) | Asking a professor to explain in one paragraph |

---

## How to Use This Notebook

- Read each **explanation cell** (grey background with 📖) before running the code below it
- **Run each code cell** with `Shift + Enter`
- Try the **🧠 Mini Quiz** sections to test your understanding
- Try the **💡 Try It Yourself** challenges to experiment

---

> **Think of this entire system as a smart research assistant:**
> You give it a document → it learns the content → you ask questions → it finds the answer and summarizes it for you.

In [ ]:
from google.colab import drive
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from transformers import pipeline

---
## 📦 Module 1 — Imports (The Toolbox)

Before writing any logic, we collect all the **tools (libraries)** we need.
Think of this like gathering your equipment before starting a science experiment.

| Import | What It Does | Why We Need It |
|--------|-------------|----------------|
| `google.colab drive` | Connects to Google Drive | So we can read files stored there |
| `gensim Word2Vec` | Trains word embeddings | Converts words → numbers |
| `gensim simple_preprocess` | Cleans and tokenizes text | Splits sentences into word lists |
| `sklearn cosine_similarity` | Measures similarity between vectors | Finds which doc matches the query |
| `numpy` | Fast math on arrays | Averages word vectors |
| `transformers pipeline` | Loads AI models from Hugging Face | Powers the summarization step |

---

> 📖 **Key Concept — Why do we need to convert words to numbers?**
>
> Computers cannot understand the word `"cat"`.
> But they can compare `[0.2, 0.8, -0.1]` with `[0.3, 0.7, -0.2]` and say "these are very similar!"
> Word2Vec is the tool that makes this translation happen.

---
**▶ Run the cell below to load all tools:**

In [ ]:
!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 4.4 MB/s eta 0:00:00


---
## 📂 Module 2 — Loading Your Document

### What is happening here?

We need to get text **out of a Word document (.docx)** so our program can read it.

```
Your .docx file                     Python
┌─────────────────────┐            ┌─────────────────────────────────┐
│ Introduction:        │   ──────▶ │ "Introduction:\nSit to stand..  │
│ Sit to stand...      │            │  Methodology:\nWe used..."      │
│ Methodology:         │            └─────────────────────────────────┘
│ We used...           │                    one big string
└─────────────────────┘
```

### Breaking down `load_docx_file(file_path)`:

```python
doc = docx.Document(file_path)        # Open the Word file
doc.paragraphs                        # List of all paragraphs
paragraph.text                        # The text inside each paragraph
"\n".join([...])                      # Join them with newline between each
```

### Breaking down `mount_and_load()`:

```python
drive.mount('/content/drive')         # Connect Google Drive to this session
file_path = "...STS paper.docx"       # Path to your file inside Drive
document_content = load_docx_file()   # Extract all text
return [document_content]             # Wrap in a list → this is the "corpus"
```

> 📖 **What is a corpus?**
> A corpus is just a **collection of documents**. Here our corpus has only 1 document — the entire paper as one string. Later we'll see why that causes a problem.

---
**▶ Run the two cells below. The second one mounts Drive and loads your document:**

In [ ]:
import docx

---
## 🔢 Module 3 — Word2Vec Embeddings (Turning Words into Numbers)

### The Big Idea

Imagine placing every word in a city on a map.
Words used in similar contexts are placed **near each other** on the map.

```
          "jogging"
             •
  "running" •   • "walking"
                          
                    • "swimming"
         
                              • "reading"
                    • "studying"  • "writing"
```

Words like `"running"` and `"jogging"` are close together.
Words like `"running"` and `"reading"` are far apart.
**Word2Vec builds this map automatically from your text.**

---

### Step A — Preprocessing with `simple_preprocess`

```python
simple_preprocess("The STS Movement is Important!")
# Output: ["the", "sts", "movement", "is", "important"]
```

It automatically:
- Lowercases everything
- Removes punctuation  
- Splits into individual words (tokens)
- Removes very short tokens (< 2 chars)

---

### Step B — Training Word2Vec

```python
Word2Vec(
    sentences = preprocessed_corpus,  # Your tokenized text
    vector_size = 100,   # Each word gets a 100-number coordinate
    window = 5,          # Look 5 words left and right for context
    min_count = 1,       # Include words that appear at least 1 time
    workers = 4          # Use 4 CPU threads to train faster
)
```

**The `window` parameter visualized:**
```
"the  patient  must  [stand]  from  the  chair  slowly"
                       ↑
               target word
      ←←← 5 words ←←←   →→→ 5 words →→→
      context words used to learn meaning of "stand"
```

> 📖 **Key Insight:** Word2Vec learns that "stand" is related to "patient", "chair", "slowly" because they always appear near each other. This gives "stand" a vector that reflects its medical/physical context in your paper.

In [ ]:
# === Step 1: Mount Google Drive and Load Document ===

def load_docx_file(file_path):
    """Load a .docx file and return its text content."""
    doc = docx.Document(file_path)
    return "\n".join([paragraph.text for paragraph in doc.paragraphs])

---
## 📐 Module 4 — Search with Cosine Similarity

### Part A — Document Vectors

Each document is a big bag of words. We need to turn the whole document into **one single vector**.

How? We **average** all the word vectors together:

```
Document: "sit to stand movement requires balance"

  "sit"      → [0.2,  0.8, -0.1]
  "stand"    → [0.3,  0.7, -0.2]
  "movement" → [0.1,  0.9,  0.0]
  "requires" → [0.4,  0.2,  0.3]
  "balance"  → [0.2,  0.6, -0.1]
               ─────────────────
  Average  → [0.24, 0.64, -0.02]   ← document vector
```

Words not found in the model get a zero vector `[0, 0, 0, ...]` so they don't affect the average.

---

### Part B — Cosine Similarity

Once everything is a vector, we measure similarity using the **angle** between two vectors.

```
         query_vector
              ↗
             /  θ (small angle = similar!)
            /
           /________▶  doc_vector


         query_vector
              ↗
             /
            /
           /
          /
         /___________________________▶ doc_vector
              θ (large angle = not similar!)
```

**Formula:**
```
                  query · doc
similarity  =  ───────────────────
               ‖query‖ × ‖doc‖
```

- Result of **1.0** = identical direction (very relevant)
- Result of **0.0** = perpendicular (unrelated)
- Result of **-1.0** = opposite direction (contradictory)

---

### Part C — The `search()` Function Walk-Through

```python
def search(query, corpus, word2vec_model, document_vectors):

    # Step 1: Turn the query string into a vector
    query_vector = np.mean([
        word2vec_model.wv[word]          # get vector for each word
        for word in query.split()        # split query into words
        if word in word2vec_model.wv     # skip unknown words
    ] or [np.zeros(...)], axis=0)        # fallback: zero vector

    # Step 2: Compare query vector against ALL document vectors
    similarities = cosine_similarity([query_vector], document_vectors)
    # Returns shape (1, N) — one similarity score per document

    # Step 3: Sort documents by score, highest first
    ranked_docs = sorted(enumerate(similarities[0]),
                         key=lambda x: x[1], reverse=True)

    # Step 4: Return (document_text, score) pairs
    return [(corpus[i], score) for i, score in ranked_docs]
```

> ⚠️ **The limitation here:** If there is only 1 document in the corpus, step 3 always returns that same document with a score near 1.0, no matter what query you type!

In [ ]:
def mount_and_load():
    """Mount Google Drive and load the document content."""
    drive.mount('/content/drive')
    file_path = "/content/drive/MyDrive/STS paper.docx"  # Update this path for the new file
    document_content = load_docx_file(file_path)
    print("Loaded document into the corpus!")
    return [document_content]

---
## 🤖 Module 5 — AI Summarization with Hugging Face BART

### What is BART?

BART stands for **Bidirectional and Auto-Regressive Transformer**.
It is a large neural network trained by Facebook AI on millions of documents.

```
Your text (up to ~1024 tokens)
          │
          ▼
   ┌─────────────────────────────┐
   │  BART Encoder               │  ← Reads and understands the whole text
   │  (like reading the paper)   │
   └─────────────────────────────┘
          │
          ▼
   ┌─────────────────────────────┐
   │  BART Decoder               │  ← Generates a shorter summary word by word
   │  (like writing a summary)   │
   └─────────────────────────────┘
          │
          ▼
   "Sit to stand capacity is a key factor..."
```

---

### How the Pipeline Works

```python
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
```

This one line:
1. Downloads the BART model (~1.6 GB) from Hugging Face
2. Loads it into memory
3. Returns a callable object — you just pass text to it

```python
summary = summarizer(
    text,
    max_length = 100,    # Output won't exceed 100 tokens
    min_length = 30,     # Output will be at least 30 tokens
    do_sample  = False   # Use greedy decoding (deterministic output)
)
```

### The `do_sample=False` Parameter Explained

| `do_sample` | Behavior | Use when |
|-------------|----------|----------|
| `False` | Always picks the most likely next word | You want consistent, factual summaries |
| `True` | Randomly samples next words | You want varied, creative outputs |

---

### The Improved Version (cell-13) adds 3 Guards:

```python
# Guard 1: empty text
if not text.strip():
    return "The text is empty and cannot be summarized."

# Guard 2: text too short for BART to summarize meaningfully
if len(text.split()) < 50:
    return "The text is too short for summarization."

# Guard 3: truncate to fit BART's input window
if len(text) > 1024:
    text = text[:1024]   # ⚠️ Note: this counts characters, not tokens!
```

> ⚠️ **Bug Alert:** The truncation uses `len(text)` which counts **characters**, but BART's limit is in **tokens** (~4 characters per token on average). This means the actual limit is ~256 tokens, cutting off most of the paper. A better approach: `text[:4096]` or use a tokenizer to count properly.

##**Step 2: Preprocess and Train Word2Vec Embeddings**
This step turns text into numbers using Word2Vec. First, preprocess_corpus cleans and tokenizes the text. Then, train_embeddings trains a Word2Vec model, creating word vectors based on context. It uses parameters like embedding size, window size, and minimum word frequency. These embeddings help capture word meanings and are later used for tasks like similarity searches.

In [ ]:
# === Step 2: Preprocess and Train Word2Vec Embeddings ===

def preprocess_corpus(corpus):
    """Preprocess text corpus into tokenized format."""
    return [simple_preprocess(doc) for doc in corpus]

def train_embeddings(corpus, embedding_dim=100):
    """Train Word2Vec embeddings on the preprocessed corpus."""
    preprocessed_corpus = preprocess_corpus(corpus)
    model = Word2Vec(sentences=preprocessed_corpus, vector_size=embedding_dim, window=5, min_count=1, workers=4)
    return model

##**Step 3: Search Functionality Using Cosine Similarity**
This step sets up search using cosine similarity. get_document_vectors turns each document into a vector by averaging its Word2Vec embeddings (using a zero vector for missing words). The search function does the same for queries, then calculates cosine similarity with all document vectors. The most relevant documents are ranked and returned with their scores, making searches more accurate and meaningful.

---
## 🔗 Module 6 — The Final Workflow (Putting It All Together)

### How `process_multiple_queries()` Works

```python
def process_multiple_queries(queries, corpus, word2vec_model, document_vectors):
    responses = {}                          # Empty dict to store results

    for query in queries:                   # Loop over each question
        print(f"Processing query: {query}")

        search_results = search(            # Step 1: Find relevant docs
            query, corpus,
            word2vec_model,
            document_vectors
        )

        if search_results:                  # Step 2: If anything was found...
            top_document = search_results[0][0]   # Take the #1 ranked doc
            summary = summarize_with_huggingface(top_document)  # Summarize it
            responses[query] = summary      # Store result

        else:
            responses[query] = "No relevant documents found."

    return responses                        # Return all query→summary pairs
```

### Data Flow for ONE Query:

```
query = "What are the key findings?"
        │
        ▼
   search()  ──────────────────────────────────────▶  [("full paper text", 0.94)]
        │                                                         │
        │                                               top_document = "full paper text"
        ▼
   summarize_with_huggingface(top_document)
        │
        ▼
   "Sit to stand capacity is a key factor..."
        │
        ▼
   responses["What are the key findings?"] = "Sit to stand capacity..."
```

### The Full Pipeline End-to-End:

```
 corpus = mount_and_load()
     │  ["entire paper as one string"]
     ▼
 word2vec_model = train_embeddings(corpus)
     │  model with vectors for every word
     ▼
 document_vectors = get_document_vectors(corpus, word2vec_model)
     │  [averaged vector for each document]
     ▼
 results = process_multiple_queries(queries, corpus, word2vec_model, document_vectors)
     │  {query: summary, query: summary, ...}
     ▼
 print results
```

In [ ]:
# === Step 3: Search Functionality Using Cosine Similarity ===

def get_document_vectors(corpus, word2vec_model):
    """Convert documents into vectors using the Word2Vec model."""
    return [
        np.mean([word2vec_model.wv[word] for word in doc if word in word2vec_model.wv] or [np.zeros(word2vec_model.vector_size)], axis=0)
        for doc in preprocess_corpus(corpus)
    ]

def search(query, corpus, word2vec_model, document_vectors):
    """Search for the most relevant documents using cosine similarity."""
    query_vector = np.mean([
        word2vec_model.wv[word] for word in query.split() if word in word2vec_model.wv
    ] or [np.zeros(word2vec_model.vector_size)], axis=0)
    similarities = cosine_similarity([query_vector], document_vectors)
    ranked_docs = sorted(enumerate(similarities[0]), key=lambda x: x[1], reverse=True)
    return [(corpus[i], score) for i, score in ranked_docs]


##**Step 4: Summarization Using Hugging Face**
This step adds text summarization using Hugging Face’s facebook/bart-large-cnn model. A summarization pipeline processes input text and generates concise summaries. The summarize_with_huggingface function applies the model with set limits (30-100 tokens) and includes error handling. This makes documents and query results easier to understand and analyze.

In [ ]:
# === Step 4: Summarization Using Hugging Face ===

# Load a Hugging Face summarization pipeline
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

def summarize_with_huggingface(text):
    """Summarize the given text using a Hugging Face model."""
    try:
        summary = summarizer(text, max_length=100, min_length=30, do_sample=False)
        return summary[0]['summary_text']
    except Exception as e:
        print(f"Error with Hugging Face summarization: {e}")
        return "Unable to generate summary."

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cpu


##**Robust Text Summarization with Input Validation and Error Handling**
This function summarizes text using a Hugging Face model. It first checks if the input is too short or empty, handling those cases smoothly. For longer text, it trims it to fit within the model’s 1024-token limit. The summarizer then generates a summary (30-100 tokens). Error handling is included to catch any issues, making the summarization process reliable and effective.

In [ ]:
def summarize_with_huggingface(text):
    """Summarize the given text using a Hugging Face model."""
    try:
        # Check for empty or very short text
        if not text.strip():
            return "The text is empty and cannot be summarized."
        if len(text.split()) < 50:  # Skip summarization for very short text
            return "The text is too short for summarization."

        # Truncate text to fit within the model's maximum token limit
        max_input_length = 1024
        if len(text) > max_input_length:
            text = text[:max_input_length]

        # Perform summarization
        summary = summarizer(text, max_length=100, min_length=30, do_sample=False)
        return summary[0]['summary_text']

    except Exception as e:
        print(f"Error with Hugging Face summarization: {e}")
        return "Unable to generate summary due to an error."

##**Final Workflow: Combining All steps**
This step ties everything together into a smooth workflow. process_multiple_queries runs searches for each query using Word2Vec and cosine similarity. The most relevant document is then summarized. If no matches are found, it returns a message instead. Results are stored in a dictionary, mapping queries to summaries or feedback, making the process efficient and automated.



In [ ]:
# === Final Workflow: Combining All Steps ===

def process_multiple_queries(queries, corpus, word2vec_model, document_vectors):
    """Process multiple search queries and summarize results."""
    responses = {}
    for query in queries:
        print(f"Processing query: {query}")
        # Perform search
        search_results = search(query, corpus, word2vec_model, document_vectors)
        if search_results:
            # Extract top document
            top_document = search_results[0][0]
            # Summarize the top document
            summary = summarize_with_huggingface(top_document)
            responses[query] = summary
        else:
            responses[query] = "No relevant documents found."
    return responses

---
## 💡 Module 7 — Try It Yourself: Challenges

Now that you understand the whole system, try improving it!

### Challenge 1 — Add More Queries (Easy)
Add 3 more queries to the list in the workflow cell and re-run to see new summaries.
```python
# Add these to the queries list:
"What datasets were used in this study?",
"What biomechanical metrics were measured?",
"Who are the target patients for this research?"
```

### Challenge 2 — Split the Document into Paragraphs (Medium)
Instead of treating the whole paper as 1 document, split it into paragraphs so search actually works:
```python
# Replace mount_and_load() with this:
def mount_and_load_paragraphs():
    drive.mount('/content/drive')
    file_path = "/content/drive/MyDrive/STS paper.docx"
    doc = docx.Document(file_path)
    # Return each paragraph as a separate document
    paragraphs = [p.text for p in doc.paragraphs if len(p.text.strip()) > 50]
    print(f"Loaded {len(paragraphs)} paragraphs as separate documents!")
    return paragraphs
```

### Challenge 3 — Fix the Truncation Bug (Medium)
Replace the character-based truncation with a proper token-aware version:
```python
# Replace the truncation line with:
max_chars = 4000   # ~1000 tokens at 4 chars/token — safe BART limit
if len(text) > max_chars:
    text = text[:max_chars]
```

### Challenge 4 — Print Similarity Scores (Easy)
Modify `process_multiple_queries()` to also print the similarity score alongside each summary:
```python
# After: top_document = search_results[0][0]
top_score = search_results[0][1]
print(f"  Similarity score: {top_score:.4f}")
```

---
## 🎓 Lesson Complete!

You have learned:
- ✅ How to extract text from Word documents with `python-docx`
- ✅ How Word2Vec converts words into meaningful number vectors
- ✅ How cosine similarity measures relevance between text and queries
- ✅ How transformer models (BART) generate abstractive summaries
- ✅ The limitations of single-document corpora in search systems
- ✅ How all 4 components connect into one end-to-end pipeline

**Next steps to make this production-quality:**
- Use `sentence-transformers` for better semantic embeddings
- Split documents into chunks (paragraphs/sections)
- Add a minimum similarity threshold to filter irrelevant results
- Use a proper tokenizer for BART input length management

In [ ]:
# === Run the Workflow ===
if __name__ == "__main__":
    # Load the corpus from Google Drive
    corpus = mount_and_load()

    # Train Word2Vec embeddings
    word2vec_model = train_embeddings(corpus)

    # Get document vectors
    document_vectors = get_document_vectors(corpus, word2vec_model)

    # Define search questions
    queries = [
        "What is the significance of the STS movement in the paper?",
        "Explain the proposed nonlinear control technique.",
        "Summarize the methodology used in the paper.",
        "What are the key findings of the study?",
        "Describe the limitations and future scope discussed in the paper."
    ]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded document into the corpus!


##**Process each query**

In [ ]:
 # Process each query
results = process_multiple_queries(queries, corpus, word2vec_model, document_vectors)


Processing query: What is the significance of the STS movement in the paper?
Processing query: Explain the proposed nonlinear control technique.
Processing query: Summarize the methodology used in the paper.
Processing query: What are the key findings of the study?
Processing query: Describe the limitations and future scope discussed in the paper.


In [ ]:
# Display results
for query, summary in results.items():
 print(f"\nQuery: {query}\nSummary: {summary}")


Query: What is the significance of the STS movement in the paper?
Summary: Sit to stand capacity is a key factor and marker in practical autonomy. Execution of sit to stand brings about physiological changes from a stable sitting position to a less stable standing position. The researchers have studied this movement in different perspectives for different understanding.

Query: Explain the proposed nonlinear control technique.
Summary: Sit to stand capacity is a key factor and marker in practical autonomy. Execution of sit to stand brings about physiological changes from a stable sitting position to a less stable standing position. The researchers have studied this movement in different perspectives for different understanding.

Query: Summarize the methodology used in the paper.
Summary: Sit to stand capacity is a key factor and marker in practical autonomy. Execution of sit to stand brings about physiological changes from a stable sitting position to a less stable standing position.

## **The model has successfully addressed all queries**